In [4]:
from utils import vector_db

vector_stores = vector_db.connect_db()

query = "How many types of BOM?"

c:\Users\wengshang.hoo\AppData\Local\miniconda3\envs\ai_sl\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading existing vector store from C:\Users\wengshang.hoo\Desktop\pp\learning-note02\rag\1. Basic Rag With Chroma\db...


**METHOD 1: Basic Similarity Search (Semantic Search/Dense Retrieval)**

Returns the top k most similar documents

In [5]:
retriever = vector_stores.as_retriever(search_kwargs={"k": 3})
docs = retriever.invoke(query)

for i, doc in enumerate(docs):
    print(f"Document {i+1}:")
    print(f"{doc.page_content}\n")

Document 1:
Multi-level BOM
A multi-level bill of materials (BOM), referred to as an indented BOM, is a bill of materials that lists the assemblies, components, and parts required to make a product in a parent-child, top-down method. It provides a display of all items that are in parent-children relationships. When an item is a sub-component, of a (parent) component, it can in-turn have its own child components, and so on. The resulting top-level BOM (item number) would include children; a mix of finished sub-assemblies, various parts and raw materials. A multi-level structure can be illustrated by a tree with several levels. In contrast, a single-level structure only consists of one level of children in components, assemblies and material.

Document 2:
A modular BOM (or variant parts list) can be displayed in the following formats:

A single-level BOM (or unit list) that displays the assembly or sub-assembly with only one level of children. Thus it displays the components directly nee

**METHOD 2: Similarity with Score Threshold**

Only returns documents above a certain similarity score

In [6]:
retriever = vector_stores.as_retriever(
    search_type="similarity_score_threshold", 
    search_kwargs={
        "k": 3,
        "score_threshold": 0.5
        }
)
docs = retriever.invoke(query)
for i, doc in enumerate(docs):
    print(f"Document {i+1}:")
    print(f"{doc.page_content}\n")

No relevant docs were retrieved using the relevance score threshold 0.5


**METHOD 3: Maximum Marginal Relevance (MMR)**

Balances relevance and diversity - avoids redundant results

In [7]:
retriever = vector_stores.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "fetch_k": 10,
        "lambda_mult": 0.5
    }
)
docs = retriever.invoke(query)
for i, doc in enumerate(docs):
    print(f"Document {i+1}:")
    print(f"{doc.page_content}\n")

Document 1:
Multi-level BOM
A multi-level bill of materials (BOM), referred to as an indented BOM, is a bill of materials that lists the assemblies, components, and parts required to make a product in a parent-child, top-down method. It provides a display of all items that are in parent-children relationships. When an item is a sub-component, of a (parent) component, it can in-turn have its own child components, and so on. The resulting top-level BOM (item number) would include children; a mix of finished sub-assemblies, various parts and raw materials. A multi-level structure can be illustrated by a tree with several levels. In contrast, a single-level structure only consists of one level of children in components, assemblies and material.

Document 2:
To decide which assembly variant of the parts or components is to be chosen, they are attributed by the product options which are the characteristic features of the product. If the options of the product build an ideal Boolean algebra,[

**Method 4: Multi Query Retrieval**

Generate queries from original question to improve the final result quality

In [8]:
from typing import List
from langchain_ollama import ChatOllama
from pydantic import BaseModel

class QueryVariations(BaseModel):
    queries: List[str]

llm = ChatOllama(model="llama3.2:1b",temperature=0.0)

In [9]:
llm_with_tools = llm.with_structured_output(QueryVariations)

prompt = f"""Generate 3 different variations of this query that would help retrieve relevant documents:

Original query: {query}

Return 3 alternative queries that rephrase or approach the same question from different angles."""

response = llm_with_tools.invoke(prompt)
query_variations = response.queries

for i, variation in enumerate(query_variations):
    print(f"Query Variation {i+1}: {variation}")

Query Variation 1: What are the various types of Business Operation Methodologies (BOM)?
Query Variation 2: How do I categorize and identify different Business Operations Methodologies?
Query Variation 3: List out all the distinct Business Operation Methodologies used in an organization


In [10]:
retriever = vector_stores.as_retriever(search_kwargs={"k": 3})
all_retrieval_results = []

for variation in query_variations:
    docs = retriever.invoke(variation)
    all_retrieval_results.append(docs)

all_retrieval_results

[[Document(id='8f2482e6-236c-4d24-8f0b-f129ead18efa', metadata={'source': 'C:\\Users\\wengshang.hoo\\Desktop\\pp\\learning-note02\\rag\\1. Basic Rag With Chroma\\docs\\bom.txt'}, page_content='A modular BOM (or variant parts list) can be displayed in the following formats:\n\nA single-level BOM (or unit list) that displays the assembly or sub-assembly with only one level of children. Thus it displays the components directly needed to make the assembly or sub-assembly.\nAn indented BOM (or structural parts list) that displays the highest-level item closest to the left margin and the components used in that item indented more to the right.[1]\nModular (planning) BOM\nA single-level BOM resolved to list the effectively needed quantities of components to produce a product (rather than to list each individual part by its logical name) is also called quantity synopsis parts list.\n\nConfigurable BOM\nA configurable bill of materials (CBOM) is a form of BOM used by industries that have multip

**Method 5: BM25 Retrieval (Keyword Search/Sparse Retrieval)**

In [11]:
%pip install rank_bm25

Note: you may need to restart the kernel to use updated packages.


In [12]:
from langchain_community.retrievers import BM25Retriever

In [14]:
for i, doc in enumerate(docs):
    print(f"Document {i+1}:")
    print(f"{doc.page_content}\n")

bm25_retriever = BM25Retriever.from_documents(docs)

print("BM25 Retriever")
print(f"-"*60)

test_docs = bm25_retriever.invoke("Tell me about Multi-level BOM")
for i, doc in enumerate(test_docs):
    print(f"Document {i+1}:")
    print(f"{doc.page_content}\n")

Document 1:
A bill of materials or product structure (sometimes bill of material, BOM or associated list) is a list of the raw materials, sub-assemblies, intermediate assemblies, sub-components, parts, and the quantities of each needed to manufacture an end product. A BOM may be used for communication between manufacturing partners or confined to a single manufacturing plant. A bill of materials is often tied to a production order whose issuance may generate reservations for components in the bill of materials that are in stock and requisitions for components that are not in stock.

The first hierarchical databases were developed for automating bills of materials for manufacturing organizations in the early 1960s. At present, this BOM is used as a database to identify the many parts and their codes in automobile manufacturing companies.

Document 2:
A modular BOM (or variant parts list) can be displayed in the following formats:

A single-level BOM (or unit list) that displays the asse

**Method 6: Hybrid Search**

In [15]:
docs = list({doc.page_content: doc for doc in test_docs + docs}.values())

docs

[Document(id='61b61137-d0bc-4805-94e2-f30335fa5c55', metadata={'source': 'C:\\Users\\wengshang.hoo\\Desktop\\pp\\learning-note02\\rag\\1. Basic Rag With Chroma\\docs\\bom.txt'}, page_content='Multi-level BOM\nA multi-level bill of materials (BOM), referred to as an indented BOM, is a bill of materials that lists the assemblies, components, and parts required to make a product in a parent-child, top-down method. It provides a display of all items that are in parent-children relationships. When an item is a sub-component, of a (parent) component, it can in-turn have its own child components, and so on. The resulting top-level BOM (item number) would include children; a mix of finished sub-assemblies, various parts and raw materials. A multi-level structure can be illustrated by a tree with several levels. In contrast, a single-level structure only consists of one level of children in components, assemblies and material.'),
 Document(id='8f2482e6-236c-4d24-8f0b-f129ead18efa', metadata={'s